# 05 · Validação do modelo no Vertex AI

Primeiro passo do Trabalho 2: provar que o `fraud_boosted` funciona **fora** do
BigQuery antes de escrever a API em cima dele.

---

### Por que este teste vem antes de tudo

O registro no Model Registry funcionou no Trabalho 1, mas registrar não é o mesmo
que implantar. A documentação diz que todo modelo pode ser registrado, e que as
restrições aparecem no deploy.

A restrição que nos interessa: na exportação, `TIMESTAMP` é aceito como **entrada**
do `TRANSFORM`, mas não como **saída**. O nosso `TRANSFORM` devolve
`transaction_ts` sem alteração, porque o `DATA_SPLIT_COL` precisa dele. Por outro
lado, a coluna do `DATA_SPLIT_COL` é excluída das features automaticamente.
Pela documentação, não dá para saber qual das duas regras vale. Só testando.

### Critério de pronto

O endpoint devolve **o mesmo `fraud_score` que o `ML.PREDICT`** para as mesmas
transações, com diferença abaixo da tolerância definida nos parâmetros.

### Custo

- **Parte A** — conferência do registro e exportação para o GCS. Não deixa
  nada ligado.
- **Parte B** — cria um endpoint, que **cobra por hora enquanto estiver
  implantado**, com ou sem tráfego. A última seção desliga tudo. Rode a Parte B
  de uma vez, do início ao fim.

In [ ]:
!pip install --quiet --upgrade google-cloud-bigquery google-cloud-aiplatform google-cloud-storage db-dtypes

In [ ]:
import json
import statistics
import time
from datetime import datetime, timezone

import pandas as pd
from google.cloud import aiplatform, bigquery, storage

In [ ]:
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Autenticado no Colab")
except ImportError:
    print("Fora do Colab: usando as credenciais do ambiente")

## Parâmetros

In [ ]:
PROJECT_ID  = "fraudflow-pdm-gps"
REGION      = "us-central1"
BUCKET_NAME = "fraudflow-pdm-gps-data"

BQ_MODEL        = f"{PROJECT_ID}.gold.fraud_boosted"
VERTEX_MODEL_ID = "fraudflow-boosted"     # VERTEX_AI_MODEL_ID do notebook 04
ENDPOINT_NAME   = "fraudflow-endpoint"

# Uma réplica fixa. O endpoint cobra pela máquina ligada, não por requisição.
# Se faltar capacidade de um tipo na região (erro 503), o deploy tenta o próximo.
# E2 primeiro: é a mais barata das três.
MACHINE_TYPES = ["e2-standard-2", "n2-standard-2", "n1-standard-2"]

AMOSTRAS_POR_CLASSE = 25      # 25 fraudes + 25 legítimas
TOLERANCIA          = 1e-4    # diferença máxima aceita entre os dois scores

client = bigquery.Client(project=PROJECT_ID, location=REGION)
gcs    = storage.Client(project=PROJECT_ID)
aiplatform.init(project=PROJECT_ID, location=REGION)
print(PROJECT_ID, REGION)

## Parte A · O modelo está onde achamos que está?

Duas visões do mesmo modelo: a do BigQuery ML e a do Model Registry. O
`CREATE OR REPLACE MODEL` substitui a versão no Registry, então a versão
listada deve ter sido criada no mesmo momento do último treino.

O campo `tipos de deploy aceitos` é o primeiro sinal: se vier vazio, o Registry
não aceita implantar este modelo num endpoint.

In [ ]:
modelo_bq = client.get_model(BQ_MODEL)

print("BigQuery ML")
print(f"  modelo           {BQ_MODEL}")
print(f"  tipo             {modelo_bq.model_type}")
print(f"  atualizado em    {modelo_bq.modified:%d/%m/%Y %H:%M} UTC")
print(f"  entrada          {[c.name for c in modelo_bq.feature_columns]}")
print(f"  rótulo           {[c.name for c in modelo_bq.label_columns]}")

modelo_vertex = aiplatform.Model(model_name=VERTEX_MODEL_ID)

print("\nModel Registry")
print(f"  recurso          {modelo_vertex.resource_name}")
print(f"  versão padrão    {modelo_vertex.version_id}")
print(f"  container        {modelo_vertex.gca_resource.container_spec.image_uri or '(gerenciado)'}")
print(f"  tipos de deploy aceitos  {[t.name for t in modelo_vertex.supported_deployment_resources_types]}")

print("\nVersões registradas")
for v in modelo_vertex.versioning_registry.list_versions():
    print(f"  v{v.version_id:<4} criada em {v.version_create_time}   aliases={list(v.version_aliases)}")

### Exportação para o GCS

É o teste rápido da restrição de `TIMESTAMP`: a exportação aplica as mesmas
regras de tipo do deploy e responde em um ou dois minutos.

Cada execução grava numa pasta nova, com data e hora no nome.

In [ ]:
destino = f"gs://{BUCKET_NAME}/models/fraud_boosted/{datetime.now(timezone.utc):%Y%m%d_%H%M%S}/"

try:
    client.query(f"EXPORT MODEL `{BQ_MODEL}` OPTIONS (URI = '{destino}')").result()
    EXPORT_OK = True
    print(f"Exportação concluída: {destino}\n")
except Exception as erro:
    EXPORT_OK = False
    print("A EXPORTAÇÃO FALHOU. Mensagem completa:\n")
    print(erro)

if EXPORT_OK:
    prefixo = destino.removeprefix(f"gs://{BUCKET_NAME}/")
    for blob in gcs.list_blobs(BUCKET_NAME, prefix=prefixo):
        print(f"  {blob.name.removeprefix(prefixo):<60} {blob.size:>10,} bytes")

**Como ler o resultado:**

- **Exportou, com uma pasta `transform/` ao lado do modelo XGBoost:** o
  pré-processamento viaja junto com o modelo. Siga para a Parte B.
- **Falhou:** leia a mensagem antes de decidir qualquer coisa. A primeira
  execução falhou, e a Parte A2 mostra como a causa foi isolada.

## Parte A2 · Diagnóstico com modelos de teste

A primeira exportação falhou com:

> *Model export with TRANSFORM only supports numerical type arguments of
> function $less, but was given type STRING.*

`$less` é o operador `<`. O único `<` entre textos no `TRANSFORM` é a correção
de aniversário da idade, que compara `FORMAT_DATE('%m%d', ...)` das duas datas.
A documentação lista o `<` como suportado, mas não diz que ele só aceita números
na exportação.

A troca é comparar `mês × 100 + dia` como número. Para datas, a ordem desse
número é a mesma da ordem do texto `'MMDD'`, então a idade não muda. A célula
seguinte confere isso na tabela inteira.

A exportação para no primeiro erro. Corrigida a idade, o `TIMESTAMP` do
`DATA_SPLIT_COL` ainda poderia aparecer. Por isso o teste treina dois modelos
mínimos e tenta exportar cada um:

| Modelo de teste | Idade | Resultado esperado | Resultado em 02/11/2026 |
|---|---|---|---|
| `export_teste_atual` | texto, a versão do T1 | **falhar**: é o controle, prova que o teste reproduz o erro | falhou com o mesmo erro |
| `export_teste_idade_num` | número, a versão atual | exportar | exportou |

O `TIMESTAMP` no recorte temporal, portanto, não bloqueia. A correção da idade
foi aplicada nos notebooks 04 e `sql/ml/`, os modelos foram retreinados e a
Parte A passou a exportar.

Os modelos usam duas semanas de dados e uma única iteração, então o custo é uma
fração mínima do treino do T1. Eles são apagados no final. **Esta parte não
precisa ser rodada de novo**: a trava abaixo impede que ela rode por engano.

In [ ]:
IDADE_TEXTO = """
  DATE_DIFF(DATE(transaction_ts), customer_dob, YEAR)
    - IF(FORMAT_DATE('%m%d', DATE(transaction_ts))
          < FORMAT_DATE('%m%d', customer_dob), 1, 0)"""

IDADE_NUMERICA = """
  DATE_DIFF(DATE(transaction_ts), customer_dob, YEAR)
    - IF(EXTRACT(MONTH FROM DATE(transaction_ts)) * 100 + EXTRACT(DAY FROM DATE(transaction_ts))
          < EXTRACT(MONTH FROM customer_dob) * 100 + EXTRACT(DAY FROM customer_dob), 1, 0)"""

# As duas fórmulas precisam dar a mesma idade em todas as linhas.
client.query(f"""
    SELECT COUNT(*)                                          AS linhas,
           COUNTIF(({IDADE_TEXTO}) != ({IDADE_NUMERICA}))   AS idades_diferentes
    FROM `{PROJECT_ID}.gold.ml_input`
""").to_dataframe()

In [ ]:
HAVERSINE = """
  2 * 6371.0 * ATAN2(
    SQRT(
      POW(SIN((merchant_lat - customer_lat) * ACOS(-1) / 360), 2)
      + COS(customer_lat * ACOS(-1) / 180)
        * COS(merchant_lat * ACOS(-1) / 180)
        * POW(SIN((merchant_long - customer_long) * ACOS(-1) / 360), 2)
    ),
    SQRT(1 -
      ( POW(SIN((merchant_lat - customer_lat) * ACOS(-1) / 360), 2)
        + COS(customer_lat * ACOS(-1) / 180)
          * COS(merchant_lat * ACOS(-1) / 180)
          * POW(SIN((merchant_long - customer_long) * ACOS(-1) / 360), 2) )
    )
  )"""

def sql_modelo_teste(nome, idade):
    """Mesmo TRANSFORM do notebook 04, variando só a idade."""
    return f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.gold.{nome}`
TRANSFORM (
  amount,
  LN(amount)                                              AS log_amount,
  category,
  EXTRACT(HOUR      FROM transaction_ts)                  AS hour,
  EXTRACT(DAYOFWEEK FROM transaction_ts)                  AS day_of_week,
  (EXTRACT(HOUR FROM transaction_ts) >= 22
   OR EXTRACT(HOUR FROM transaction_ts) <= 3)             AS is_night,
  {idade}                                                 AS age,
  {HAVERSINE}                                             AS distance_km,
  city_pop,
  transaction_ts,
  is_fraud
)
OPTIONS (
  MODEL_TYPE               = 'BOOSTED_TREE_CLASSIFIER',
  MAX_ITERATIONS           = 1,
  INPUT_LABEL_COLS         = ['is_fraud'],
  DATA_SPLIT_METHOD        = 'SEQ',
  DATA_SPLIT_COL           = 'transaction_ts',
  DATA_SPLIT_EVAL_FRACTION = 0.20
) AS
SELECT * EXCEPT (transaction_id, split_key)
FROM `{PROJECT_ID}.gold.ml_input`
WHERE transaction_ts < TIMESTAMP '2019-01-15'
"""


VARIANTES = {
    "export_teste_atual":     IDADE_TEXTO,
    "export_teste_idade_num": IDADE_NUMERICA,
}

RODAR_DIAGNOSTICO = False   # já rodado em 02/11/2026; ver a tabela acima

if not RODAR_DIAGNOSTICO:
    raise RuntimeError("Diagnóstico não executado: mude RODAR_DIAGNOSTICO para True se quiser repetir.")

resultados = []
for nome, idade in VARIANTES.items():
    modelo = f"{PROJECT_ID}.gold.{nome}"
    uri = f"gs://{BUCKET_NAME}/models/export_teste/{nome}/"
    try:
        client.query(sql_modelo_teste(nome, idade)).result()
        client.query(f"EXPORT MODEL `{modelo}` OPTIONS (URI = '{uri}')").result()
        resultados.append((nome, "EXPORTOU", ""))
    except Exception as erro:
        resultados.append((nome, "FALHOU", str(erro).split("\n")[0][-250:]))
    finally:
        client.delete_model(modelo, not_found_ok=True)
    print(f"{nome:<24} {resultados[-1][1]}")

pd.set_option("display.max_colwidth", 250)
pd.DataFrame(resultados, columns=["modelo de teste", "resultado", "erro"])

## Parte A3 · O artefato exportado

O enunciado aceita duas formas de servir o modelo na API: consultar um endpoint
do Vertex AI ou carregar o modelo dentro do próprio container. A exportação da
Parte A gerou esse artefato: o modelo XGBoost (`model.bst`), o `TRANSFORM`
como TensorFlow SavedModel (`transform/`) e um preditor do próprio BigQuery ML
(`main.py` e `xgboost_predictor-0.1.tar.gz`).

Esta parte só baixa a exportação mais recente e mostra como o preditor funciona,
para avaliar se a API pode carregar o modelo sem endpoint. Não cria nada na
nuvem e não tem custo.

In [ ]:
import tarfile
from pathlib import Path

BASE_EXPORT = "models/fraud_boosted/"
pastas = sorted({b.name.removeprefix(BASE_EXPORT).split("/")[0]
                 for b in gcs.list_blobs(BUCKET_NAME, prefix=BASE_EXPORT)} - {""})
EXPORT_DIR = f"{BASE_EXPORT}{pastas[-1]}/"
LOCAL = Path("/content/fraud_boosted_export")

for blob in gcs.list_blobs(BUCKET_NAME, prefix=EXPORT_DIR):
    if blob.name.endswith("/"):
        continue
    destino = LOCAL / blob.name.removeprefix(EXPORT_DIR)
    destino.parent.mkdir(parents=True, exist_ok=True)
    blob.download_to_filename(destino)

print(f"exportação usada: gs://{BUCKET_NAME}/{EXPORT_DIR}\n")
for caminho in sorted(LOCAL.rglob("*")):
    if caminho.is_file():
        print(f"  {caminho.relative_to(LOCAL)}")

In [ ]:
def mostrar(titulo, texto):
    print(f"{'=' * 78}\n{titulo}\n{'=' * 78}\n{texto}\n")


mostrar("main.py", (LOCAL / "main.py").read_text())
mostrar("assets/model_metadata.json", (LOCAL / "assets/model_metadata.json").read_text())
mostrar("explanation_metadata.json", (LOCAL / "explanation_metadata.json").read_text())

with tarfile.open(LOCAL / "xgboost_predictor-0.1.tar.gz") as pacote:
    for membro in pacote.getmembers():
        if membro.isfile() and membro.name.endswith((".py", ".txt", ".cfg")):
            mostrar(f"xgboost_predictor · {membro.name}", pacote.extractfile(membro).read().decode())

## Parte B · Endpoint

> **A partir daqui há custo por hora.** Mude `CONFIRMO_CUSTO` para `True` e rode
> até a seção **Desligar**, no final. O deploy leva de 10 a 20 minutos.

Se o Colab desconectar, rode de novo as células iniciais (até **Parâmetros**), a
trava e o deploy. Um deploy já concluído é detectado e não se repete.

In [ ]:
CONFIRMO_CUSTO = False

if not CONFIRMO_CUSTO:
    raise RuntimeError("Parte B não executada: mude CONFIRMO_CUSTO para True.")

In [ ]:
from google.api_core.exceptions import ServiceUnavailable

# Busca o modelo aqui mesmo: se o Colab desconectar, basta rodar as células
# iniciais, a trava e esta.
modelo_vertex = aiplatform.Model(model_name=VERTEX_MODEL_ID)

# Reaproveita um endpoint com o mesmo nome, se já existir: evita pagar por dois.
existentes = aiplatform.Endpoint.list(filter=f'display_name="{ENDPOINT_NAME}"')
endpoint = existentes[0] if existentes else aiplatform.Endpoint.create(display_name=ENDPOINT_NAME)
print(f"endpoint  {endpoint.resource_name}")
print(f"modelo    {VERTEX_MODEL_ID} v{modelo_vertex.version_id}")

if endpoint.list_models():
    print("Já existe um modelo implantado neste endpoint; o deploy foi pulado.")
else:
    for machine_type in MACHINE_TYPES:
        print(f"\nimplantando em {machine_type}...")
        inicio = time.perf_counter()
        try:
            modelo_vertex.deploy(
                endpoint=endpoint,
                machine_type=machine_type,
                min_replica_count=1,
                max_replica_count=1,
                traffic_percentage=100,
            )
            print(f"Deploy concluído em {(time.perf_counter() - inicio) / 60:.1f} min")
            break
        except ServiceUnavailable as erro:
            print(f"sem capacidade para {machine_type} agora: {str(erro)[:120]}")
        except Exception as erro:
            print("O DEPLOY FALHOU. Mensagem completa:\n")
            print(erro)
            raise
    else:
        raise RuntimeError("Nenhum tipo de máquina disponível: tente de novo em alguns minutos.")

for implantado in endpoint.list_models():
    print(f"  implantado: {implantado.display_name}  versão {implantado.model_version_id}  ({implantado.dedicated_resources.machine_spec.machine_type})")

## Amostra de teste

Transações da **janela de validação** do treino — os 20% finais por tempo, a
mesma janela do notebook 04. O holdout (`fraudTest`) continua fora: ele é a fonte
do streaming e da avaliação final.

A amostra é balanceada para cobrir os dois lados da decisão. A ordem por
`FARM_FINGERPRINT` deixa a amostra igual a cada execução.

In [ ]:
CAMPOS = [
    "transaction_ts", "customer_dob",
    "customer_lat", "customer_long", "merchant_lat", "merchant_long",
    "amount", "category", "city_pop",
]

SQL_AMOSTRA = f"""
WITH corte AS (
  SELECT APPROX_QUANTILES(transaction_ts, 5)[OFFSET(4)] AS inicio_validacao
  FROM `{PROJECT_ID}.gold.ml_input`
),
validacao AS (
  SELECT m.*
  FROM `{PROJECT_ID}.gold.ml_input` AS m, corte
  WHERE m.transaction_ts >= corte.inicio_validacao
)
(SELECT transaction_id, {", ".join(CAMPOS)}, is_fraud
 FROM validacao WHERE is_fraud = 1
 ORDER BY FARM_FINGERPRINT(transaction_id) LIMIT {AMOSTRAS_POR_CLASSE})
UNION ALL
(SELECT transaction_id, {", ".join(CAMPOS)}, is_fraud
 FROM validacao WHERE is_fraud = 0
 ORDER BY FARM_FINGERPRINT(transaction_id) LIMIT {AMOSTRAS_POR_CLASSE})
"""

amostra = client.query(SQL_AMOSTRA).to_dataframe()
print(f"{len(amostra)} transações  ·  {amostra.is_fraud.sum()} fraudes")
amostra.head()

## Referência: `ML.PREDICT`

O score que o BigQuery dá para as mesmas transações. É contra ele que o
endpoint será comparado. O rótulo não entra na predição.

In [ ]:
SQL_REFERENCIA = f"""
SELECT
  transaction_id,
  (SELECT p.prob
   FROM UNNEST(predicted_is_fraud_probs) AS p
   WHERE CAST(p.label AS STRING) = '1')                   AS score_bq
FROM ML.PREDICT(
  MODEL `{BQ_MODEL}`,
  (SELECT transaction_id, {", ".join(CAMPOS)}
   FROM `{PROJECT_ID}.gold.ml_input`
   WHERE transaction_id IN UNNEST(@ids))
)
"""

config = bigquery.QueryJobConfig(query_parameters=[
    bigquery.ArrayQueryParameter("ids", "STRING", amostra.transaction_id.tolist()),
])
referencia = client.query(SQL_REFERENCIA, job_config=config).to_dataframe()
amostra = amostra.merge(referencia, on="transaction_id")
amostra[["transaction_id", "transaction_ts", "is_fraud", "score_bq"]].head()

## Formato da requisição

A documentação não diz como o endpoint espera receber `TIMESTAMP` e `DATE`. Em
vez de supor, o teste manda uma fraude e uma transação legítima em cada formato
candidato. O formato só é aceito se o score bater com o `ML.PREDICT`.

Isso importa: um formato lido com fuso errado não dá erro, só muda a hora. E a
hora decide `is_night`, a feature mais forte do modelo.

In [ ]:
FORMATOS_TS = {
    "texto simples": "%Y-%m-%d %H:%M:%S",
    "ISO com T":     "%Y-%m-%dT%H:%M:%S",
    "com UTC":       "%Y-%m-%d %H:%M:%S UTC",
    "ISO com Z":     "%Y-%m-%dT%H:%M:%SZ",
}


def montar_instancia(linha, formato_ts):
    """Os nove campos do contrato, em tipos nativos do JSON."""
    return {
        "transaction_ts": pd.Timestamp(linha.transaction_ts).strftime(formato_ts),
        "customer_dob":   pd.Timestamp(linha.customer_dob).strftime("%Y-%m-%d"),
        "customer_lat":   float(linha.customer_lat),
        "customer_long":  float(linha.customer_long),
        "merchant_lat":   float(linha.merchant_lat),
        "merchant_long":  float(linha.merchant_long),
        "amount":         float(linha.amount),
        "category":       str(linha.category),
        "city_pop":       int(linha.city_pop),
    }


def score_de(predicao):
    """Probabilidade da classe '1' numa predição do endpoint.

    O BigQuery ML nomeia as chaves a partir do rótulo (is_fraud_probs,
    is_fraud_values...). A busca pelo sufixo evita depender do nome exato.
    """
    probs   = next(v for k, v in predicao.items() if k.endswith("_probs"))
    valores = [str(v) for v in next(v for k, v in predicao.items() if k.endswith("_values"))]
    return float(probs[valores.index("1")])


teste = pd.concat([amostra[amostra.is_fraud == 1].head(1), amostra[amostra.is_fraud == 0].head(1)])
FORMATO_TS = None
resposta_exemplo = None

for nome, formato in FORMATOS_TS.items():
    instancias = [montar_instancia(l, formato) for l in teste.itertuples()]
    try:
        resposta = endpoint.predict(instances=instancias)
        scores = [score_de(p) for p in resposta.predictions]
        diferencas = [abs(s - r) for s, r in zip(scores, teste.score_bq)]
        bate = max(diferencas) < TOLERANCIA
        print(f"{nome:<14} aceito   scores={[round(s, 6) for s in scores]}   "
              f"diferença máx={max(diferencas):.2e}   {'BATE' if bate else 'não bate'}")
        if bate and FORMATO_TS is None:
            FORMATO_TS, resposta_exemplo = formato, resposta
    except Exception as erro:
        print(f"{nome:<14} recusado: {str(erro)[:200]}")

print(f"\nreferência ML.PREDICT: {[round(s, 6) for s in teste.score_bq]}")
print(f"formato escolhido: {FORMATO_TS!r}")

In [ ]:
# Resposta crua do endpoint, para documentar o contrato.
if resposta_exemplo is not None:
    print(f"versão do modelo que respondeu: {resposta_exemplo.model_version_id}\n")
    print(json.dumps(resposta_exemplo.predictions[0], indent=2, ensure_ascii=False))

## Comparação completa

Todas as transações da amostra, numa única requisição.

In [ ]:
assert FORMATO_TS is not None, "Nenhum formato bateu com o ML.PREDICT: veja a célula anterior."

instancias = [montar_instancia(l, FORMATO_TS) for l in amostra.itertuples()]
resposta = endpoint.predict(instances=instancias)

amostra["score_vertex"] = [score_de(p) for p in resposta.predictions]
amostra["diferenca"] = (amostra.score_vertex - amostra.score_bq).abs()
amostra["hora"] = pd.to_datetime(amostra.transaction_ts).dt.hour

fora = amostra[amostra.diferenca >= TOLERANCIA]
print(f"transações comparadas   {len(amostra)}")
print(f"diferença máxima        {amostra.diferenca.max():.2e}")
print(f"acima da tolerância     {len(fora)}")
print()
print("CRITÉRIO ATINGIDO: endpoint e ML.PREDICT concordam." if fora.empty
      else "CRITÉRIO NÃO ATINGIDO: veja as linhas abaixo.")

(fora if not fora.empty else amostra)[
    ["transaction_id", "hora", "category", "amount", "is_fraud", "score_bq", "score_vertex", "diferenca"]
].sort_values("score_bq", ascending=False).head(15)

## Latência

Medida daqui do Colab até o endpoint, uma transação por chamada. Serve de
ordem de grandeza: a latência que vale para a entrega é a da API no Cloud Run,
medida na etapa da API.

In [ ]:
tempos_ms = []
for linha in amostra.head(21).itertuples():
    inicio = time.perf_counter()
    endpoint.predict(instances=[montar_instancia(linha, FORMATO_TS)])
    tempos_ms.append((time.perf_counter() - inicio) * 1000)

primeira, demais = tempos_ms[0], sorted(tempos_ms[1:])
print(f"primeira chamada   {primeira:7.0f} ms")
print(f"mediana            {statistics.median(demais):7.0f} ms")
print(f"pior das 20        {demais[-1]:7.0f} ms")

## O contrato, para o README

Exemplo real de requisição e resposta. É a referência para a API do Cloud Run.

In [ ]:
exemplo = amostra.sort_values("score_bq", ascending=False).iloc[0]
print("REQUISIÇÃO")
print(json.dumps({"instances": [montar_instancia(exemplo, FORMATO_TS)]}, indent=2, ensure_ascii=False))
print("\nRESPOSTA")
print(json.dumps(endpoint.predict(instances=[montar_instancia(exemplo, FORMATO_TS)]).predictions[0],
                 indent=2, ensure_ascii=False))

## Desligar

Remove o modelo do endpoint e apaga o endpoint. O modelo continua no Registry:
implantar de novo é só rodar a Parte B outra vez.

**Rode esta seção mesmo que alguma célula acima tenha falhado.**

In [ ]:
for ep in aiplatform.Endpoint.list(filter=f'display_name="{ENDPOINT_NAME}"'):
    ep.undeploy_all()
    ep.delete()
    print(f"apagado: {ep.resource_name}")

restantes = aiplatform.Endpoint.list(filter=f'display_name="{ENDPOINT_NAME}"')
print(f"endpoints restantes com este nome: {len(restantes)}")

---

**Guarde este notebook com as saídas** (Arquivo → Salvar uma cópia no GitHub, ou
baixe o `.ipynb`): a comparação e a latência são evidências do PDF de entrega.

**Próximo:** contratos do streaming e da API.